# SDG 13 — Unified BERTopic Analysis (Targets 13.1, 13.2, 13.3)

This notebook combines BERTopic analysis for **all 3 SDG 13 targets** in a single file.
The pipeline runs automatically 3 times and produces separate outputs per target.


> **How to use**: Replace `[INPUT_YOUR_FILE_HERE]` in `TARGETS_CONFIG` with your file paths,
> then run all cells from top to bottom.

### Output per Target
- `topic_info_SDG13X.csv` — topic list & top words
- `documents_with_topics_SDG13X.csv` — documents with topic assignment
- `topic_distribution_SDG13X.csv` — topic distribution summary
- `coherence_per_topic_SDG13X.csv` — coherence score per topic
- `topic_barchart_SDG13X.html`, `topic_map_SDG13X.html`, `topic_hierarchy_SDG13X.html`, `topic_heatmap_SDG13X.html`
- `bertopic_model_SDG13X/` — saved model



## 1. Install Required Packages

In [ ]:
!pip install bertopic sentence-transformers umap-learn hdbscan pandas openpyxl plotly scikit-learn gensim -q

## 2. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import re
from collections import Counter

from bertopic import BERTopic
from bertopic.vectorizers import ClassTfidfTransformer
from bertopic.representation import KeyBERTInspired, MaximalMarginalRelevance
from sentence_transformers import SentenceTransformer
from umap import UMAP
from hdbscan import HDBSCAN
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics import silhouette_score

# For topic coherence
from gensim.corpora.dictionary import Dictionary
from gensim.models.coherencemodel import CoherenceModel

import warnings
warnings.filterwarnings('ignore')

print("✓ All libraries imported successfully")

## 3. Global Model Parameters (Standardized Across All Targets)

These pipeline parameters are **identical for all three targets (13.1, 13.2, 13.3)** so the
results are directly comparable. The only things that vary across targets are the
**input data**, **whitelist** (protected terms), **seed topics**, and **target-specific
extra stopwords** — defined in `TARGETS_CONFIG` below.

In [ ]:
# ============================================================
# GLOBAL MODEL CONFIGURATION (shared across SDG 13.1, 13.2, 13.3)
# ============================================================

MODEL_CONFIG = {

    # 1. Document Embedding
    "embedding": {
        "model_name": "all-MiniLM-L6-v2",
    },

    # 2. Dimensionality Reduction (UMAP)
    "umap": {
        "n_neighbors": 15,
        "n_components": 5,
        "min_dist": 0.0,
        "metric": "cosine",
        "random_state": 42,
    },

    # 3. Topic Clustering (HDBSCAN)
    "hdbscan": {
        "min_cluster_size": 15,
        "min_samples": 5,
        "metric": "euclidean",
        "cluster_selection_method": "eom",   # Excess of Mass
        "prediction_data": True,
    },

    # 4. Vectorization (CountVectorizer)
    "vectorizer": {
        "ngram_range": (1, 2),               # Unigrams + Bigrams
        "min_df": 2,
        "max_df": 0.95,
    },

    # 5. Topic Representation
    "ctfidf": {
        "seed_multiplier": 3,                # Boost seed words 3x
        "reduce_frequent_words": True,
    },
    "representation": {
        "mmr_diversity": 0.3,                # MMR diversity parameter
    },

    # 6. BERTopic top-level
    "bertopic": {
        "nr_topics": "auto",
        "top_n_words": 10,
        "calculate_probabilities": True,
        "verbose": True,
    },
}

print("✓ Global model parameters loaded")
print(f"  • Embedding:      {MODEL_CONFIG['embedding']['model_name']}")
print(f"  • UMAP:           n_neighbors={MODEL_CONFIG['umap']['n_neighbors']}, "
      f"n_components={MODEL_CONFIG['umap']['n_components']}, metric={MODEL_CONFIG['umap']['metric']}")
print(f"  • HDBSCAN:        min_cluster_size={MODEL_CONFIG['hdbscan']['min_cluster_size']}, "
      f"metric={MODEL_CONFIG['hdbscan']['metric']}, selection={MODEL_CONFIG['hdbscan']['cluster_selection_method']}")
print(f"  • Vectorizer:     ngram_range={MODEL_CONFIG['vectorizer']['ngram_range']}, "
      f"min_df={MODEL_CONFIG['vectorizer']['min_df']}, max_df={MODEL_CONFIG['vectorizer']['max_df']}")
print(f"  • Representation: cTF-IDF with seed_multiplier={MODEL_CONFIG['ctfidf']['seed_multiplier']}, "
      f"KeyBERTInspired + MMR(diversity={MODEL_CONFIG['representation']['mmr_diversity']})")

## 4. Common Stopwords (Shared Across All Targets)

Generic stopwords that apply to all three targets. Target-specific stopwords are added
separately in `TARGETS_CONFIG`.

⚠️ **Important**: These stopwords do **NOT** remove SDG 13 key terms such as
*resilience, adaptation, climate, disaster, awareness, education, policy, governance*.

In [ ]:
# ============================================================
# COMMON STOPWORDS (shared across SDG 13.1, 13.2, 13.3)
# ============================================================

# Names and people (noise)
COMMON_PERSON_NAMES = {
    "malhotra", "daby", "elmore", "ellie", "flynn", "perkins", "seema",
    "trump", "biden", "harris", "kamala", "greta", "thunberg",
    "leonardo", "dicaprio", "schwarzenegger", "gore"
}

# Generic noise (basic English filler + common topical noise observed across targets)
COMMON_GENERIC_NOISE = {
    "thing", "stuff", "way", "lot", "really", "good", "great", "nice", "best", 'woman', 'gender',
    'girl', 'woman girl', 'auspol', 'liberal', 'lnp', 'liberal party', 'party', 'auspol net', 'zero auspol',
    'insider', 'coalition', 'election', 'ai', 'monitoring', 'infrastructure', 'ai climate', 'data center', 'survelliance',
    'digital', 'change ai', 'center', 'ley', 'seat', 'libs', 'wilson', 'tim', 'Inpdyingparty', 'cyclone', 'tim wilson', 'sussan ley',
    'sussan', 'datacenters', 'data centre', 'artificialintelligence', 'artificial intelligence', 'data', 'building',
    'net zero', 'netzero', 'design', 'labour', 'scam', 'youth', 'children', 'young', 'youthforclimate', 'climateeducation', 'young people',
    'pakistan', 'preparedness', 'punjab', 'child', 'man', 'mountain', 'ocean', 'river', 'airplane', 'bag', 'umbrella', 'glass', 'bottle',
    'father', 'sibling', 'event', 'account', 'area', 'object', 'place', 'food', 'plant', 'museum',
    'city', 'country', 'person', 'function', 'table', 'chair', 'home', 'street', 'bike', 'car', 'plane', 'apolistic', 'schwarzeneger',
    'day', 'hour', 'month', 'year', 'number', 'book', 'paper', 'tree', 'leaf', 'flower', 'fruit',
    'water', 'earth', 'wind', 'village', 'road', 'space', 'floor', 'ceiling', 'light', 'dark',
    'cold', 'hot', 'sky', 'cloud', 'rain', 'sun', 'moon', 'star', 'forest', 'beach', 'island',
    'hill', 'window', 'door', 'stairs', 'garden', 'park', 'lake', 'field',
    'soil', 'stone', 'sand', 'seed', 'root', 'branch', 'twig', 'stem', 'shape', 'color', 'size',
    'prince', 'william', 'king', 'charles', 'earthshot', 'son', 'sharma', 'prize', 'climate', 'pope', 'francis',
    'do', 'does', 'did', 'doing', 'having', 'with', 'becomes', 'become', 'becoming', 'am', 'are', 'is',
    'specie', 'marine', 'fishery', 'variation', 'animal', 'said', 'genetic', 'frog', 'bay', 'monday',
    'nbc', 'announce', 'arrived', 'preserve', 'ireland', 'tourism', 'accf', 'austria', 'african',
    'africa', 'fleming',
    'gulf', 'maine', 'vulnerability', 'science', 'also', 'blooded', 'fish', 'temperature', 'model', 'sea', 'puddle',
    'dubose', 'mental', 'health', 'anxiety', 'adolescent', 'impact', 'australian', 'madagascar', 'eco',
    'concern', 'stress', 'depression', 'level', 'jasper', 'site', 'heritage', 'conservancy', 'funding', 'burn',
    'altering', 'united', 'nature', 'montana',
    'and', 'or', 'but', 'because', 'while', 'however', 'as', 'so', 'that', 'which', 'for',
    'on', 'at', 'between', 'before', 'after', 'during', 'about', 'within', 'under', 'above', 'without',
    'than', 'these', 'those', 'all', 'each', 'more', 'less', 'many', 'few', 'this', 'it', 'its', 'their',
    'the', 'a', 'an', 'in', 'of', 'to', 'from', 'by', 'into', 'towards', 'against',
    'not', 'be', 'being', 'been', 'was', 'were', 'will', 'have', 'has', 'had',
    'creation', 'care', 'schwaezenegger', 'welby', 'apostolic', 'gloninger', 'paris', 'plaintiff', 'trial',
    'seeley', 'judge', 'healthful', 'california', 'supreme', 'enbridge', 'pipeline', 'hole', 'gdss', 'melissa',
    'jamaica', 'huriccanemelissa', 'gust', 'mph', 'brunt', 'destructive', 'category', 'beautiful', 'reclaw',
    'vegan', 'petition', 'lentil', 'salted', 'doughnut', 'mash', 'caramel', 'gluten', 'mini', 'potato',
    'virus', 'zombie', 'thawed', 'french', 'frozen', 'scientist', 'infectious', 'research', 'permafrost', 'threat',
    'allergy', 'pollen', 'season', 'allergic', 'airway', 'longer', 'symptom', 'steiner', 'higher', 'sinus',
    'million', 'article', 'term', 'sks', 'roundup', 'box', 'displayed', 'completely', 'coral', 'reef',
    'new', 'author', 'study', 'ancient', 'weekly', 'turn', 'find', 'control', 'panel', 'israel', 'gaza', 'haaretz',
    'flame', 'speech', 'israeli', 'mohammed', 'bizarrely', 'jesuralem',
    'walk', 'run', 'sit', 'stand', 'eat', 'drink', 'look', 'hear', 'speak', 'say', 'write', 'read', 'think',
    'feel', 'know', 'remember', 'forget', 'live', 'die', 'come', 'go', 'arrive', 'leave', 'enter', 'exit',
    'open', 'close', 'take', 'give', 'push', 'pull', 'grab', 'hold', 'help', 'carry', 'start', 'finish',
    'continue', 'stop', 'increase', 'decrease', 'change', 'create', 'generate', 'reduce', 'measure', 'improve',
    'uk', 'honour', 'summit', 'world', 'duke', 'cop', 'restoration', 'list', 'planet', 'royal', 'alok',
    'honour list', 'british', 'crisis', 'church', 'bishop', 'si', 'laudato si', 'laudato', 'interfaith', 'one', 'guinea', 'people', 'god',
    'anesthetic', 'montreal protocol', 'use', 'substance', 'near zero', 'montreal', 'metered dose', 'metered', 'pmdis',
    'dr net', 'asset value', 'net asset', 'dr', 'value', 'asset', 'net', 'announcement', 'ey', 'announcement transmitted',
    'clwd ln', 'swept away', 'mana', 'northern', 'swept', 'pas', 'near', 'away', 'least', 'yana', 'region',
    'indian', 'army', 'construction', 'indian army',
    'bringing weather', 'photo picture', 'whats', 'whats happening', 'update find', 'find whats', 'enjoy award',
    'picture taken', 'keep date', 'winning photo', 'catch business', 'date update', 'award winning', 'enjoy', 'taken catch',
    'lord', 'created', 'awardee', 'award', 'occasion', 'participated international', 'sandworld', 'lord shiva',
    'shiva', 'rakhi', 'shri', 'weymouth', 'terrorism', 'shri awardee',
    'music', 'festival', 'vatican', 'pontiff', 'cardinal', 'harry', 'time', 'athens', 'baird', 'ucits', 'msci',
    'reactor', 'doomsday', 'atomic', 'worker', 'video', 'county', 'climber',
    'catholic', 'leo', 'action', 'xiv', 'global', 'boston', 'encyclical', 'environmental', 'faith', 'rome', 'diego', 'american',
    'court', 'lawsuit', 'law', 'legal', 'company', 'state', 'case', 'international', 'fuel', 'fossil',
    'nye', 'lemon', 'twitter', 'icj', 'ruling', 'exxon',
    'sustainability', 'green', 'conservation', 'environment', 'eco-friendly', 'recycle', 'waste',
    'reuse', 'renewable', 'carbon', 'carbon footprint', 'climate adaptation',
    'global warming', 'job', 'cost', 'dump', 'business', 'price', 'zero zero', 'Lcoy', 'learning',
    'unicef', 'voices', 'curriculum', 'punjabfloods'
}

# Time expressions (too generic)
COMMON_TIME_NOISE = {
    "monday", "tuesday", "wednesday", "thursday", "friday", "saturday", "sunday",
    "morning", "afternoon", "evening",
    'foods', 'mad', 'kg', 'miliband', 'energy bill', 'bill', 'carbonfootprint', 'tax',
    'ed', 'scrap', 'students', 'schools', 'kids', 'youthleadership', 'school', 'planning', 'pakistanfloods',
    'cleaner', 'storage', 'mrandmissclimateke', 'circulareconomy', 'every', 'collective', 'ambition',
    'bold', 'brazil', 'count', 'foodsecurity', 'address', 'treeplanting', 'community', 'scale',
    'indigenous', 'led', 'local', 'strategy', 'idea', 'practice'
}

# Social media jargon
COMMON_SOCIAL_MEDIA = {
    "rt", "via", "cc", "dm", "followback", "retweet",
    'britain', 'madness', 'pay', 'removal',
    'offset', 'icoy', 'lcoy', 'training', 'classroom', 'empower', 'punjabfloods',
    'sector', 'investment', 'warming', 'rising', 'related', 'floodsinpakistan',
    'rise', 'national', 'actnow need', 'needed', 'wait', 'register', 'taking',
    'india', 'finance', 'together', 'plan', 'vulnerable', 'need', 'innovation'
}

# Combine into the common base
COMMON_STOPWORDS = (
    COMMON_PERSON_NAMES
    | COMMON_GENERIC_NOISE
    | COMMON_TIME_NOISE
    | COMMON_SOCIAL_MEDIA
)

print(f"✓ Common stopwords loaded: {len(COMMON_STOPWORDS)} terms")

## 5. Per-Target Configuration

Each target only differs in:
- **Input file & text column** — different Excel file per target (replace path here!)
- **Whitelist** — protected key terms that must be preserved
- **Seed topics** — guiding seed groups for BERTopic
- **Extra stopwords** — additional target-specific noise terms
- **Search keywords** — keyword groups for downstream document filtering

> 💡 Replace `[INPUT_YOUR_FILE_HERE]` with the actual path to your Excel file
> (e.g. `'BIGRAM 1.xlsx'` or `'/content/BIGRAM 1.xlsx'`).

In [ ]:
# ============================================================
# TARGETS_CONFIG — only target-specific settings live here
# (model parameters are global, see MODEL_CONFIG above)
# ============================================================

TARGETS_CONFIG = {

    # =====================================================================
    # SDG 13.1 — Climate Resilience & Adaptive Capacity
    # =====================================================================
    "13.1": {
        "name": "Climate Resilience & Adaptive Capacity",
        "description": "Strengthen resilience and adaptive capacity to climate-related hazards and natural disasters in all countries",
        "input_file": "[INPUT_YOUR_FILE_HERE]",   # ← Replace with: 'BIGRAM 1.xlsx' or full path
        "text_column": "TEXT",
        "output_suffix": "SDG131",

        "whitelist": {
            # Core SDG 13.1 terms
            "resilience", "resilient", "adaptation", "adaptive", "capacity",
            "vulnerability", "vulnerable",
            # Disaster management
            "disaster", "hazard", "risk", "emergency", "preparedness", "response",
            "mitigation", "recovery", "warning", "alert", "forecast",
            # Climate impacts
            "extreme", "weather", "storm", "drought", "flood", "flooding",
            "cyclone", "hurricane", "typhoon", "wildfire", "heatwave",
            "heat", "precipitation", "rainfall", "temperature",
            # Infrastructure
            "infrastructure", "building", "construction", "engineering",
            # Core climate
            "climate", "warming", "carbon", "emission", "greenhouse",
            "environmental", "environment", "sustainable", "sustainability",
            # Action & planning
            "action", "planning", "strategy", "measure", "intervention",
            "protection", "safeguard", "strengthen",
            # Geographic/community
            "community", "local", "coastal", "island", "mountain", "urban", "rural"
        },

        "seed_topics": [
            ["resilience", "adaptation", "adaptive", "capacity", "strengthen", "recovery"],
            ["disaster", "hazard", "risk", "emergency", "preparedness", "response"],
            ["extreme", "weather", "storm", "drought", "flood", "cyclone", "hurricane"],
            ["warning", "monitoring", "forecast", "alert", "prediction", "surveillance"],
            ["infrastructure", "planning", "building", "design", "engineering", "construction"]
        ],

        "extra_stopwords": set(),  # 13.1 uses common stopwords only

        "search_groups": {
            "resilience": "resilience|resilient|adaptation|adaptive",
            "disaster":   "disaster|flood|drought|storm|cyclone|hurricane|wildfire|emergency",
        },
    },

    # =====================================================================
    # SDG 13.2 — Policy Integration & Governance
    # =====================================================================
    "13.2": {
        "name": "Policy Integration & Governance",
        "description": "Integrate climate change measures into national policies, strategies and planning",
        "input_file": "[INPUT_YOUR_FILE_HERE]",   # ← Replace with: 'BIGRAM 2.xlsx' or full path
        "text_column": "TEXT",
        "output_suffix": "SDG132",

        "whitelist": {
            # Core SDG 13.2 terms
            "policy", "governance", "integration", "framework", "regulation",
            "institutional", "mainstreaming", "commitment", "legislation",
            "planning", "strategy", "climate strategy", "national adaptation plan", "sectoral policy",
            # Climate action planning & policy
            "nationally determined contributions", "ndc", "climate action", "mitigation",
            "adaptation", "policy coherence", "policy reform", "climate legislation", "green governance",
            # Legislative frameworks
            "law", "carbon pricing", "emission reduction", "carbon tax", "environmental policy",
            "climate law", "sustainable development", "green economy", "environmental regulations",
            # Institutional frameworks
            "institutional capacity", "capacity building", "government", "sectoral integration", "implementation",
            "partnership", "public sector", "private sector", "multilateral agreements", "international cooperation",
            # Action & development
            "development agenda", "climate policy", "climate governance",
            "action plans", "policy instruments", "sector policies", "climate policies", "regulatory frameworks",
            # Geographic/community
            "local", "national", "regional", "international", "coastal", "urban", "rural", "developing countries"
        },

        "seed_topics": [
            ["policy", "governance", "integration", "framework", "legislation", "strategy", "mainstreaming"],
            ["climate action", "climate strategy", "ndc", "national adaptation plan", "climate policy", "mitigation", "adaptation"],
            ["institutional capacity", "capacity building", "government", "implementation", "institutional frameworks", "public sector"],
            ["policy coherence", "policy reform", "sectoral integration", "regulatory frameworks", "climate-responsive policies", "climate governance"],
            ["international cooperation", "multilateral agreements", "climate governance", "green economy", "sustainable development", "partnership"]
        ],

        "extra_stopwords": {
            "vital", "protect", "protecting", "shaping", "revolution",
            "life", "efficiency", "source", "sustainableenergy",
            "justenergytransition", "solutions"
        },

        "search_groups": {
            "policy":     "policy|policies|legislation|regulation|law",
            "governance": "governance|institutional|government|framework|implementation",
        },
    },

    # =====================================================================
    # SDG 13.3 — Education, Awareness & Capacity Building
    # =====================================================================
    "13.3": {
        "name": "Education, Awareness & Capacity Building",
        "description": "Improve education, awareness-raising and human and institutional capacity on climate change mitigation, adaptation, impact reduction and early warning",
        "input_file": "[INPUT_YOUR_FILE_HERE]",   # ← Replace with: 'BIGRAM 3.xlsx' or full path
        "text_column": "TEXT",
        "output_suffix": "SDG133",

        "whitelist": {
            # Core SDG 13.3 terms
            "awareness", "education", "learning", "literacy", "knowledge", "training",
            "capacity", "skills", "empowerment",
            # Public awareness and outreach
            "communication", "outreach", "campaign", "engagement", "information", "media",
            "participation", "public", "society",
            # Operational systems and preparedness
            "preparedness", "response", "early warning", "monitoring", "alert", "forecast",
            "emergency", "disaster", "resilience",
            # Institutional and human capacity
            "institutional", "capacity building", "human resources", "workshop", "institution",
            "organizational", "development",
            # Climate impacts and action
            "climate", "change", "adaptation", "mitigation", "sustainability", "sustainable",
            "environment", "environmental",
            # Climate action and planning
            "action", "strategy", "measure", "intervention", "protection", "safeguard",
            "strengthen", "policy", "planning",
            # Geographic/community
            "community", "local", "coastal", "urban", "rural", "island", "mountain",
            "vulnerable", "region",
            # Bigrams
            "climate resilience", "climate literacy", "risk communication",
            "community action", "training programs", "resilience training"
        },

        "seed_topics": [
            ["climate education", "climate awareness", "climate literacy", "public education", "sustainability education", "awareness campaigns", "climate action awareness"],
            ["public outreach", "communication strategies", "media campaigns", "community engagement", "information dissemination", "climate risk communication"],
            ["capacity building", "institutional training", "human resources", "organizational development", "institutional capacity", "training programs", "workshops"],
            ["early warning", "disaster preparedness", "climate risk", "mitigation strategies", "response systems", "emergency preparedness", "climate impact reduction"],
            ["operational systems", "climate monitoring", "climate resilience", "emergency response", "climate information services", "risk reduction", "climate adaptation"],
        ],

        # 13.3-specific extra noise terms (from previous modeling iterations)
        "extra_stopwords": {
            # Person names additions
            "august", "since", "around", "real", "join", "today", "fflive", "absolutrly",
            "actioning", "even", "waiting", "solution sustainable", "repo", "driven", "cycle",
            # Generic noise additions
            "ever", "coming", "forward", "reality", "serve", "true", "toward",
            "energy sustainable", "clean", "highest", "plants", "aviation", "inaction",
            "implement", "solve", "drive", "fellow", "energy susbainable", "tech", "project",
            "big", "please", "solutions sustainable",
            # Time noise additions
            "september", "october", "vital", "protect", "protecting", "shaping", "revolution",
            "across", "inperson", "forum", "small", "addis", "better", "point", "right",
            "call", "behind", "news", "deniers", "denier", "causing", "narrative",
            "leader", "minister", "accountability", "jobs", "private", "dao", "transparency",
            "conference", "energy energy", "energy powered", "energyefficiency", "battery",
            "bills", "proud", "moment", "individual", "aacjication", "actionnow", "vision", "focus",
            # Social media additions
            "energy sustainable", "energy", "goals", "th", "us", "enough", "record", "displaced",
            "solution sustainability", "based", "tomorrow", "sustainable power", "save", "high",
            "generation", "sustainable solution", "future future", "crypto", "nothing", "show",
            "face", "heard", "generations", "stay", "north", "changed", "brings", "aka", "safer",
            "web", "immigration", "keep", "cannot", "life", "efficiency", "source", "sustainableenergy",
            "warn", "times", "urban", "last", "already", "diferrence", "mean", "absolutely",
            "makes", "frequent", "house", "healthier", "build", "zero watch", "driver", "power",
            "cut", "difference", "affect", "behaviour", "main", "distant", "lead", "critical",
            "rate", "happening", "step", "gap", "ensure", "pushing", "path", "possible", "follow",
            "counts", "ideas", "matter", "past", "everyone", "greatest", "happen", "cause",
            "getting", "page", "steps", "story", "suppo", "reach", "inpakistan", "week", "another",
            "fight", "turned", "turning", "yet", "justenergytransition", "solutions",
            "like", "affected", "changing", "living", "learn", "dont", "caused", "still",
            "induced", "floods", "america", "hope", "emissions", "leaders", "supply", "first",
            "choice", "biggest", "share", "latest", "talking", "tipping", "fix", "always",
            "driving", "changes", "fake"
        },

        "search_groups": {
            "education": "education|awareness|literacy|learning|knowledge|training",
            "capacity":  "capacity|empowerment|skills|workshop|outreach|engagement",
        },
    },
}

print("✓ TARGETS_CONFIG loaded")
for tid, cfg in TARGETS_CONFIG.items():
    n_seeds = sum(len(s) for s in cfg["seed_topics"])
    print(f"  • SDG {tid} — {cfg['name']}")
    print(f"      whitelist: {len(cfg['whitelist'])} terms | seed words: {n_seeds} | extra stopwords: {len(cfg['extra_stopwords'])}")

## 6. Helper Functions

Includes the **coherence score calculator** (`calculate_coherence`) which computes both
`c_v` and `c_npmi` using gensim's `CoherenceModel`.

In [ ]:
def clean_text(text):
    """Basic cleaning while preserving important climate terms."""
    if not isinstance(text, str):
        return ""
    text = text.lower()
    text = re.sub(r'http\S+|www\S+|https\S+', '', text)   # URLs
    text = re.sub(r'\S+@\S+', '', text)                    # emails
    text = re.sub(r'@\w+', '', text)                        # mentions
    text = re.sub(r'#(\w+)', r'\1', text)                  # hashtags → keep word
    text = re.sub(r'\s+', ' ', text).strip()                # whitespace
    return text


def get_topic_distribution(topic_model, input_docs):
    """Build a topic distribution summary DataFrame."""
    doc_info = topic_model.get_document_info(input_docs)
    if 'Topic' not in doc_info.columns:
        raise ValueError("doc_info must contain a 'Topic' column.")

    topic_counts = doc_info['Topic'].value_counts().sort_index()
    total_docs = len(doc_info)
    total_docs_wo_noise = len(doc_info[doc_info['Topic'] != -1])

    topic_keywords = {}
    for topic_id in topic_counts.index:
        if topic_id != -1:
            topic_words = topic_model.get_topic(topic_id)
            topic_keywords[topic_id] = (
                ", ".join([w for w, _ in topic_words[:5]]) if topic_words else "No keywords"
            )
        else:
            topic_keywords[topic_id] = "Noise/Outliers"

    summary = pd.DataFrame({
        'Topic': topic_counts.index,
        'n_doc': topic_counts.values,
        'percentage': (topic_counts.values / total_docs) * 100,
        'keywords': [topic_keywords.get(t, "No keywords") for t in topic_counts.index]
    })
    summary['percentage_wo_noise'] = summary.apply(
        lambda row: (row['n_doc'] / total_docs_wo_noise * 100) if row['Topic'] != -1 else 0,
        axis=1
    )
    return summary


def check_topic_relevance(topic_model, topics, whitelist):
    """Count how many topics contain whitelisted (target-relevant) terms."""
    relevant_topics = []
    for topic_id in set(topics):
        if topic_id == -1:
            continue
        topic_words = topic_model.get_topic(topic_id)
        relevant_count = sum(1 for word, _ in topic_words if word in whitelist)
        if relevant_count > 0:
            relevant_topics.append({
                'topic_id': topic_id,
                'relevant_words': relevant_count,
                'top_words': [w for w, _ in topic_words[:5]],
                'doc_count': sum(1 for t in topics if t == topic_id)
            })
    return relevant_topics


def calculate_coherence(topic_model, docs, topics, top_n=10, metrics=('c_v', 'c_npmi')):
    """
    Compute topic coherence scores using gensim's CoherenceModel.

    Tokenization is aligned with BERTopic's vectorizer so that bigrams produced by the
    cTF-IDF representation are also recognised in the coherence corpus.

    Parameters
    ----------
    topic_model : BERTopic
        Fitted BERTopic instance.
    docs : list[str]
        Documents passed to topic_model.fit_transform.
    topics : list[int]
        Topic assignments per document (output of fit_transform).
    top_n : int
        Number of top words per topic to evaluate.
    metrics : tuple[str]
        Coherence metric names ('c_v', 'c_npmi', 'c_uci', 'u_mass').

    Returns
    -------
    dict with:
        - 'overall' : {metric: float}             — mean coherence across all topics
        - 'per_topic' : pandas.DataFrame          — per-topic coherence scores
    """
    # Use BERTopic's vectorizer analyzer to align tokenization (handles unigrams + bigrams)
    vectorizer = topic_model.vectorizer_model
    analyzer = vectorizer.build_analyzer()
    tokens = [analyzer(doc) for doc in docs]

    # Build dictionary and BoW corpus
    dictionary = Dictionary(tokens)
    corpus = [dictionary.doc2bow(tk) for tk in tokens]

    # Collect topic words (skip outlier -1)
    topic_ids_sorted = sorted([t for t in set(topics) if t != -1])
    topic_words = []
    for tid in topic_ids_sorted:
        words = [w for w, _ in topic_model.get_topic(tid)[:top_n]]
        topic_words.append(words)

    if not topic_words:
        return {'overall': {m: None for m in metrics}, 'per_topic': pd.DataFrame()}

    overall = {}
    per_topic_scores = {}

    for metric in metrics:
        try:
            cm = CoherenceModel(
                topics=topic_words,
                texts=tokens,
                corpus=corpus,
                dictionary=dictionary,
                coherence=metric
            )
            overall[metric] = cm.get_coherence()
            per_topic_scores[metric] = cm.get_coherence_per_topic()
        except Exception as e:
            print(f"      ⚠️ Coherence '{metric}' failed: {e}")
            overall[metric] = None
            per_topic_scores[metric] = [None] * len(topic_ids_sorted)

    # Build per-topic DataFrame
    per_topic_df = pd.DataFrame({
        'Topic': topic_ids_sorted,
        'top_words': [", ".join(w[:5]) for w in topic_words],
    })
    for metric in metrics:
        per_topic_df[metric] = per_topic_scores.get(metric, [None] * len(topic_ids_sorted))

    return {'overall': overall, 'per_topic': per_topic_df}


print("✓ Helper functions defined")

## 7. Main Pipeline

In [ ]:
def run_bertopic_for_target(target_id, config, model_config, common_stopwords):
    """
    Run the full BERTopic pipeline for one SDG 13 target.

    Returns a dict containing: model, topics, probs, df, embeddings, summary,
    relevant_topics, coherence.
    """
    suffix = config["output_suffix"]
    name = config["name"]

    print("\n" + "█" * 80)
    print(f"  RUNNING SDG {target_id}  —  {name}")
    print("█" * 80)

    # ----- 1. LOAD DATA -----
    input_file = config["input_file"]
    if input_file == "[INPUT_YOUR_FILE_HERE]":
        raise ValueError(
            f"⚠️ Input file is not set for SDG {target_id}! "
            f"Edit TARGETS_CONFIG['{target_id}']['input_file']."
        )

    print(f"\n[1/9] Loading data from: {input_file}")
    df = pd.read_excel(input_file)
    text_col = config["text_column"]
    print(f"      Shape: {df.shape} | Text column: '{text_col}'")

    # ----- 2. CLEAN TEXT -----
    print(f"[2/9] Cleaning text...")
    df['cleaned_text'] = df[text_col].apply(clean_text)
    df = df[df['cleaned_text'].str.len() > 10].reset_index(drop=True)
    print(f"      Documents after cleaning: {len(df)}")

    # ----- 3. BUILD STOPWORDS -----
    custom_stopwords = common_stopwords | config["extra_stopwords"]
    print(f"[3/9] Stopwords: {len(custom_stopwords)} (common={len(common_stopwords)} + target-extras={len(config['extra_stopwords'])})")
    print(f"      Whitelist (protected): {len(config['whitelist'])} terms")

    # ----- 4. SEED WORDS (flatten) -----
    seed_words = [w for topic_list in config["seed_topics"] for w in topic_list]
    print(f"[4/9] Seed topics: {len(config['seed_topics'])} groups | {len(seed_words)} seed words")

    # ----- 5. CONFIGURE PIPELINE COMPONENTS -----
    print(f"[5/9] Configuring BERTopic components...")
    embedding_model = SentenceTransformer(model_config["embedding"]["model_name"])

    umap_model = UMAP(**model_config["umap"])

    hdbscan_model = HDBSCAN(**model_config["hdbscan"])

    vectorizer_model = CountVectorizer(
        stop_words=list(custom_stopwords),
        **model_config["vectorizer"]
    )

    ctfidf_model = ClassTfidfTransformer(
        seed_words=seed_words,
        **model_config["ctfidf"]
    )

    representation_model = {
        "KeyBERT": KeyBERTInspired(),
        "MMR": MaximalMarginalRelevance(diversity=model_config["representation"]["mmr_diversity"])
    }

    topic_model = BERTopic(
        embedding_model=embedding_model,
        umap_model=umap_model,
        hdbscan_model=hdbscan_model,
        vectorizer_model=vectorizer_model,
        ctfidf_model=ctfidf_model,
        representation_model=representation_model,
        **model_config["bertopic"]
    )

    # ----- 6. FIT MODEL -----
    print(f"[6/9] Generating embeddings + fitting model...")
    texts = df['cleaned_text'].tolist()
    embeddings = embedding_model.encode(texts, show_progress_bar=True)
    topics, probs = topic_model.fit_transform(texts, embeddings)

    n_topics = len(set(topics)) - (1 if -1 in topics else 0)
    n_outliers = sum(1 for t in topics if t == -1)
    print(f"      ✓ Topics found: {n_topics} | Outliers: {n_outliers}")

    # ----- 7. COHERENCE SCORE -----
    print(f"[7/9] Computing coherence scores (c_v, c_npmi)...")
    coherence = calculate_coherence(
        topic_model, texts, topics,
        top_n=model_config["bertopic"].get("top_n_words", 10),
        metrics=('c_v', 'c_npmi')
    )
    overall_cv = coherence['overall'].get('c_v')
    overall_npmi = coherence['overall'].get('c_npmi')
    print(f"      ✓ c_v    = {overall_cv:.4f}" if overall_cv is not None else "      ✗ c_v    = N/A")
    print(f"      ✓ c_npmi = {overall_npmi:.4f}" if overall_npmi is not None else "      ✗ c_npmi = N/A")

    # Save per-topic coherence
    if not coherence['per_topic'].empty:
        coherence['per_topic'].to_csv(f'coherence_per_topic_{suffix}.csv', index=False)
        print(f"      ✓ coherence_per_topic_{suffix}.csv")

    # ----- 8. SAVE TABULAR OUTPUTS -----
    print(f"[8/9] Saving CSVs...")

    topic_info = topic_model.get_topic_info()
    topic_info.to_csv(f'topic_info_{suffix}.csv', index=False)

    df['topic'] = topics
    df['topic_probability'] = [max(p) if hasattr(p, '__len__') else p for p in probs]
    topic_labels = {}
    for i in set(topics):
        if i != -1:
            top_words = topic_model.get_topic(i)
            topic_labels[i] = "_".join([w for w, _ in top_words[:3]])
        else:
            topic_labels[i] = "outlier"
    df['topic_label'] = df['topic'].map(topic_labels)
    df.to_csv(f'documents_with_topics_{suffix}.csv', index=False)

    summary = get_topic_distribution(topic_model, texts)
    summary.to_csv(f'topic_distribution_{suffix}.csv', index=False)

    print(f"      ✓ topic_info_{suffix}.csv")
    print(f"      ✓ documents_with_topics_{suffix}.csv")
    print(f"      ✓ topic_distribution_{suffix}.csv")

    # ----- 9. SAVE VISUALIZATIONS -----
    print(f"[9/9] Generating visualizations...")
    try:
        fig = topic_model.visualize_barchart(top_n_topics=min(8, n_topics), n_words=10,
                                             title=f"Topic Words Score — SDG {target_id}")
        fig.write_html(f'topic_barchart_{suffix}.html')
        print(f"      ✓ topic_barchart_{suffix}.html")
    except Exception as e:
        print(f"      ⚠️ barchart failed: {e}")

    try:
        fig = topic_model.visualize_topics(title=f"Topic Similarity Map — SDG {target_id}")
        fig.write_html(f'topic_map_{suffix}.html')
        print(f"      ✓ topic_map_{suffix}.html")
    except Exception as e:
        print(f"      ⚠️ topic map failed: {e}")

    try:
        hierarchical_topics = topic_model.hierarchical_topics(texts)
        fig = topic_model.visualize_hierarchy(hierarchical_topics=hierarchical_topics,
                                              title=f"Topic Hierarchy — SDG {target_id}")
        fig.write_html(f'topic_hierarchy_{suffix}.html')
        print(f"      ✓ topic_hierarchy_{suffix}.html")
    except Exception as e:
        print(f"      ⚠️ hierarchy failed: {e}")

    try:
        fig = topic_model.visualize_heatmap(title=f"Topic Similarity Heatmap — SDG {target_id}",
                                            n_clusters=min(5, n_topics))
        fig.write_html(f'topic_heatmap_{suffix}.html')
        print(f"      ✓ topic_heatmap_{suffix}.html")
    except Exception as e:
        print(f"      ⚠️ heatmap failed: {e}")

    # ----- RELEVANCE CHECK -----
    relevant = check_topic_relevance(topic_model, topics, config["whitelist"])
    print(f"\n  → Topics with SDG {target_id} terms: {len(relevant)} / {n_topics} "
          f"({len(relevant)/max(n_topics,1)*100:.1f}%)")

    # ----- TOP WORDS PER TOPIC (display) -----
    print(f"\n  Top words per topic (first 5):")
    for tid in sorted(set(topics))[:6]:
        if tid == -1:
            continue
        words = topic_model.get_topic(tid)[:8]
        word_str = ", ".join(w for w, _ in words)
        print(f"    Topic {tid}: {word_str}")

    print(f"\n✓ SDG {target_id} pipeline complete\n")

    return {
        "target_id": target_id,
        "config": config,
        "model": topic_model,
        "topics": topics,
        "probs": probs,
        "df": df,
        "embeddings": embeddings,
        "umap_model": umap_model,
        "summary": summary,
        "relevant_topics": relevant,
        "coherence": coherence,
        "n_topics": n_topics,
        "n_outliers": n_outliers,
    }


print("✓ Main pipeline function defined")

## 8. Run Pipeline for All 3 Targets

The loop runs SDG 13.1, 13.2, and 13.3 sequentially.

In [ ]:
# ============================================================
# Run pipeline for all targets
# ============================================================
RESULTS = {}

for target_id, config in TARGETS_CONFIG.items():
    try:
        RESULTS[target_id] = run_bertopic_for_target(
            target_id=target_id,
            config=config,
            model_config=MODEL_CONFIG,
            common_stopwords=COMMON_STOPWORDS
        )
    except Exception as e:
        print(f"\n❌ ERROR for SDG {target_id}: {e}")
        print(f"   Skipping this target, moving on...\n")
        RESULTS[target_id] = None

print("\n" + "═" * 80)
print(f"  PIPELINE COMPLETE — {sum(1 for r in RESULTS.values() if r)} of {len(TARGETS_CONFIG)} targets succeeded")
print("═" * 80)

## 9. Exploration: Filter Documents by Keyword Group

Find documents that match the keyword patterns defined in `search_groups` for each target.

In [ ]:
for target_id, result in RESULTS.items():
    if result is None:
        continue

    config = result["config"]
    df = result["df"]
    suffix = config["output_suffix"]

    print(f"\n{'─' * 80}")
    print(f"  SDG {target_id} — Keyword Search")
    print('─' * 80)

    for group_name, pattern in config["search_groups"].items():
        matched = df[df['cleaned_text'].str.contains(pattern, case=False, na=False)]
        pct = len(matched) / len(df) * 100 if len(df) else 0
        print(f"\n  [{group_name.upper()}] pattern: '{pattern}'")
        print(f"      Documents matched: {len(matched)} ({pct:.1f}%)")
        if len(matched) > 0:
            top_topics = matched['topic'].value_counts().head(3)
            print(f"      Top topics: {dict(top_topics)}")

## 10. Quality Metrics — Coherence, Silhouette, Topic Diversity

For each target, this section reports:
- **Topic Coherence** (`c_v` and `c_npmi`) — semantic interpretability of the discovered topics
  - `c_v`: ranges roughly 0–1; higher = more interpretable. >0.55 is generally good.
  - `c_npmi`: ranges -1 to 1; higher = more coherent. >0.1 is generally good.
- **Silhouette Score** — cluster separation quality (-1 to 1; higher = better)
- **Topic Diversity** — count of unique top words across topics

In [ ]:
for target_id, result in RESULTS.items():
    if result is None:
        continue

    topics = result["topics"]
    embeddings = result["embeddings"]
    topic_model = result["model"]
    coherence = result["coherence"]
    suffix = result["config"]["output_suffix"]

    print(f"\n  SDG {target_id} ({suffix})")
    print('─' * 60)

    # ----- COHERENCE -----
    cv = coherence['overall'].get('c_v')
    npmi = coherence['overall'].get('c_npmi')
    if cv is not None:
        cv_label = "✓ Good" if cv > 0.55 else ("~ Fair" if cv > 0.40 else "⚠️ Low")
        print(f"    Coherence c_v:    {cv:.4f}   {cv_label}")
    else:
        print(f"    Coherence c_v:    N/A")
    if npmi is not None:
        npmi_label = "✓ Good" if npmi > 0.1 else ("~ Fair" if npmi > 0.0 else "⚠️ Low")
        print(f"    Coherence c_npmi: {npmi:.4f}   {npmi_label}")
    else:
        print(f"    Coherence c_npmi: N/A")

    # ----- SILHOUETTE (excluding outliers) -----
    mask = np.array(topics) != -1
    if mask.sum() > 0 and len(set(np.array(topics)[mask])) > 1:
        score = silhouette_score(embeddings[mask], np.array(topics)[mask])
        sil_label = "✓ Good separation" if score > 0.1 else "⚠️ Topics may overlap"
        print(f"    Silhouette Score: {score:.4f}   {sil_label}")
    else:
        print(f"    Silhouette Score: not computable (insufficient clusters)")

    # ----- TOPIC DIVERSITY -----
    all_words = set()
    for tid in set(topics):
        if tid != -1:
            words = [w for w, _ in topic_model.get_topic(tid)]
            all_words.update(words)
    n_topics = len(set(topics)) - (1 if -1 in topics else 0)
    if n_topics > 0:
        print(f"    Unique topic words: {len(all_words)}")
        print(f"    Avg words/topic:   {len(all_words)/n_topics:.1f}")

## 11. Cross-Target Summary

In [ ]:
print("\n" + "═" * 80)
print("  SDG 13 — UNIFIED ANALYSIS SUMMARY")
print("═" * 80)

summary_rows = []
for target_id, result in RESULTS.items():
    if result is None:
        summary_rows.append({
            "Target": f"SDG {target_id}",
            "Name": TARGETS_CONFIG[target_id]["name"],
            "Status": "❌ FAILED",
            "Documents": "—",
            "Topics": "—",
            "Outliers": "—",
            "Relevant Topics": "—",
            "c_v": "—",
            "c_npmi": "—",
        })
        continue

    n_topics = result["n_topics"]
    cv = result["coherence"]["overall"].get("c_v")
    npmi = result["coherence"]["overall"].get("c_npmi")
    summary_rows.append({
        "Target": f"SDG {target_id}",
        "Name": result["config"]["name"],
        "Status": "✓ DONE",
        "Documents": len(result["df"]),
        "Topics": n_topics,
        "Outliers": result["n_outliers"],
        "Relevant Topics": f"{len(result['relevant_topics'])}/{n_topics}",
        "c_v": f"{cv:.4f}" if cv is not None else "N/A",
        "c_npmi": f"{npmi:.4f}" if npmi is not None else "N/A",
    })

summary_df = pd.DataFrame(summary_rows)
print()
print(summary_df.to_string(index=False))

# Save the cross-target summary
summary_df.to_csv("SDG13_cross_target_summary.csv", index=False)
print("\n✓ Saved: SDG13_cross_target_summary.csv")

# List all generated files
print("\n" + "─" * 80)
print("  Files generated per target:")
print("─" * 80)
for target_id, result in RESULTS.items():
    if result is None:
        continue
    suffix = result["config"]["output_suffix"]
    print(f"\n  SDG {target_id} ({suffix}):")
    for fname in [
        f"topic_info_{suffix}.csv",
        f"documents_with_topics_{suffix}.csv",
        f"topic_distribution_{suffix}.csv",
        f"coherence_per_topic_{suffix}.csv",
        f"topic_barchart_{suffix}.html",
        f"topic_map_{suffix}.html",
        f"topic_hierarchy_{suffix}.html",
        f"topic_heatmap_{suffix}.html",
    ]:
        print(f"    • {fname}")

## 12. Save Models (Optional)

In [ ]:
# Save all trained models
for target_id, result in RESULTS.items():
    if result is None:
        continue
    suffix = result["config"]["output_suffix"]
    model_path = f"bertopic_model_{suffix}"
    try:
        result["model"].save(model_path, serialization="pytorch")
        print(f"✓ Saved: {model_path}")
    except Exception as e:
        print(f"⚠️ Failed to save {model_path}: {e}")

# To load a saved model later:
# model = BERTopic.load("bertopic_model_SDG131")

## 13. Inspect Individual Target (Optional)

Use this cell to explore the results for one target in more detail.

In [ ]:
# Choose which target to inspect
TARGET_TO_INSPECT = "13.1"   # change to "13.2" or "13.3" as needed

result = RESULTS.get(TARGET_TO_INSPECT)
if result is None:
    print(f"❌ SDG {TARGET_TO_INSPECT} is not available in RESULTS.")
else:
    topic_model = result["model"]
    topics = result["topics"]
    df = result["df"]
    config = result["config"]
    coherence = result["coherence"]

    print(f"\n  SDG {TARGET_TO_INSPECT} — {config['name']}")
    print("─" * 70)
    print(f"  Total documents: {len(df):,}")
    print(f"  Topics found:    {result['n_topics']}")
    print(f"  Outliers:        {result['n_outliers']:,}")
    cv = coherence['overall'].get('c_v')
    npmi = coherence['overall'].get('c_npmi')
    print(f"  Coherence c_v:    {cv:.4f}" if cv is not None else "  Coherence c_v:    N/A")
    print(f"  Coherence c_npmi: {npmi:.4f}" if npmi is not None else "  Coherence c_npmi: N/A")

    print(f"\n  Topic distribution:")
    print(result["summary"].to_string(index=False))

    print(f"\n  Per-topic coherence:")
    if not coherence['per_topic'].empty:
        print(coherence['per_topic'].to_string(index=False))

    print(f"\n  Top words per topic:")
    for tid in sorted(set(topics)):
        if tid == -1:
            continue
        words = topic_model.get_topic(tid)
        if not words:
            continue
        doc_count = sum(1 for t in topics if t == tid)
        relevant = [w for w, _ in words if w in config["whitelist"]]
        print(f"\n    Topic {tid} ({doc_count} docs):")
        for w, s in words[:10]:
            marker = " ★" if w in config['whitelist'] else ""
            print(f"      {w:30s} {s:.4f}{marker}")
        if relevant:
            print(f"      → Whitelist matches: {', '.join(relevant[:5])}")